# 00 — Setup & cadrage

**Rôle :** préparer l'environnement, créer l'arborescence, poser la provenance et le checkpoint légal. **Ne télécharge aucune donnée.**

**Place dans le pipeline :** premier notebook à exécuter. Les suivants supposent que cette structure existe.

**Cellule 1 — Imports.** On charge les libs standard du pipeline et on affiche leurs versions (traçabilité).

In [1]:
import sys, json, os
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

print("Python :", sys.version.split()[0])
print("pandas :", pd.__version__)
print("Exécuté le :", datetime.now(timezone.utc).isoformat(timespec="seconds"))

Python : 3.12.13
pandas : 3.0.3
Exécuté le : 2026-06-23T21:55:24+00:00


**Cellule 2 — Racine du projet.** On la détecte en remontant jusqu'à un repère (`requirements.txt` ou `.git`). Au besoin, fixez `PROJECT_ROOT` à la main.

In [2]:
def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start  # repli : dossier courant

PROJECT_ROOT = find_project_root(Path.cwd())
print("Racine du projet :", PROJECT_ROOT)

Racine du projet : /Users/lemairealice/Downloads/Jupiter/semaine 1


**Cellule 3 — Chemins centraux.** Tous les dossiers de données et de rapports sont définis ici, une seule fois.

In [3]:
DATA            = PROJECT_ROOT / "data"
REFERENCE       = DATA / "reference"
RAW_HOUSE_INDEX = DATA / "raw" / "house" / "index"
RAW_HOUSE_PDF   = DATA / "raw" / "house" / "ptr_pdfs"
PROC_HOUSE      = DATA / "processed" / "house"
PROC_SENATE     = DATA / "processed" / "senate"
PROCESSED       = DATA / "processed"
EXT_SENATE      = DATA / "external" / "senate_openset"
EXT_QUIVER      = DATA / "external" / "quiver"
AUDIT           = DATA / "audit"
REPORTS         = PROJECT_ROOT / "reports"

ALL_DIRS = [REFERENCE, RAW_HOUSE_INDEX, RAW_HOUSE_PDF, PROC_HOUSE, PROC_SENATE,
            PROCESSED, EXT_SENATE, EXT_QUIVER, AUDIT, REPORTS]

**Cellule 4 — Création de l'arborescence.** Idempotent : relançable sans risque.

In [4]:
for d in ALL_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    print("ok :", d.relative_to(PROJECT_ROOT))

ok : data/reference
ok : data/raw/house/index
ok : data/raw/house/ptr_pdfs
ok : data/processed/house
ok : data/processed/senate
ok : data/processed
ok : data/external/senate_openset
ok : data/external/quiver
ok : data/audit
ok : reports


**Cellule 5 — Paramètres du pipeline.** Bornes temporelles (ère électronique) et politesse réseau (pause entre requêtes).

In [5]:
HOUSE_YEARS  = range(2013, 2027)   # House : index XML fiable à partir de 2013
SENATE_YEARS = range(2012, 2027)   # Sénat : ère électronique (STOCK Act, 2012+)
REQUEST_PAUSE_S = 0.6              # pause entre requêtes (politesse / rate-limit)
USER_AGENT = "Ramify-QIS research (contact: <ton-email>)"  # identification honnête

print("House :", HOUSE_YEARS.start, "->", HOUSE_YEARS.stop - 1)
print("Sénat :", SENATE_YEARS.start, "->", SENATE_YEARS.stop - 1)

House : 2013 -> 2026
Sénat : 2012 -> 2026


**Cellule 6 — Provenance & réutilisation (transparence).**

Tout le code substantiel vit *dans* ces notebooks (inspectable) ; **aucune API boîte noire n'est source canonique**. Réutilisations :

| Élément | Source | Licence | Usage |
|---|---|---|---|
| Parsing PDF House (regex marqueurs, nettoyage octets nuls, dédup) | seralifatih/congress-trading-pipeline | MIT | porté en Python dans nos cellules |
| Index House + client Quiver | projet existant | interne | repris / adapté |
| Socle Sénat (ère électronique) | Senate Stock Watcher + kadoa (JSON) | open data / MIT | ingéré (données transparentes) |
| Identité + commissions | unitedstates/congress-legislators | domaine public | table de référence (BioGuideID) |
| Vérification | Quiver (compte dispo) | abonnement | notebook de validation seulement |

**Cellule 7 — Checkpoint légal §105(c) (à lever avec le directeur).**

⚠️ Les déclarations du Congrès (House *et* Sénat) sont publiques, mais l'Ethics in Government Act (§105(c)) **interdit leur usage commercial**, hors médias diffusant au public. Pour un asset manager (Ramify/Valhyr), à clarifier avec le directeur / le juridique **avant toute mise en production**. On avance techniquement, mais le drapeau reste ouvert. *(Ce n'est pas un avis juridique.)*

**Cellule 8 — Token Quiver (vérification seulement).** On vérifie sa présence sans jamais l'afficher. Quiver ne sert qu'au notebook de validation et n'entre **jamais** dans la table finale.

In [6]:
quiver_present = bool(os.environ.get("QUIVER_API_TOKEN", "").strip())
print("QUIVER_API_TOKEN présent :", quiver_present)
if not quiver_present:
    print("-> optionnel ici ; requis uniquement pour 08_validation_quiver.")

QUIVER_API_TOKEN présent : False
-> optionnel ici ; requis uniquement pour 08_validation_quiver.


**Cellule 9 — Périmètre & plafond de complétude.**

Cette semaine : une table de transactions **House + Sénat**, propre et structurée. Plafond assumé : **ère électronique, niveau Member**, avec un **backlog documenté** pour le papier/scanné ancien (surtout Sénat < 2015). Hors-scope cette semaine : OCR du backlog, mapping GICS→ETF (S2), stratégie & backtest (S3-S4).

Setup terminé ✅ — passez à **`01_reference_legislators`**.